# Offline Policy Evaluation


### Introduction

This notebook demonstrates the use of offline policy evaluation for MABs.

### Objectives

#### Evaluation:

Evaluate the performance of a MAB using multiple offline policy estimators.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

from pybandits.cmab import CmabBernoulliCC
from pybandits.offline_policy_evaluator import OfflinePolicyEvaluator

%load_ext autoreload
%autoreload 2

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Generate data

We first generate a binarly labeled data set, with a two dimensional feature space, and is not lineraly seprabale.
We then split the data set to a training data setm and a test data set.

In [2]:
n_samples = 1000
n_actions = 2
n_batches = 3
n_rewards = 1
n_groups = 2
n_features = 3

In [3]:
unique_actions = [f"a{i}" for i in range(n_actions)]
action_ids = np.random.choice(unique_actions, n_samples * n_batches)
batches = [i for i in range(n_batches) for _ in range(n_samples)]
rewards = [np.random.randint(2, size=(n_samples * n_batches)) for _ in range(n_rewards)]
action_true_rewards = {(a, r): np.random.rand() for a in unique_actions for r in range(n_rewards)}
true_rewards = [
    np.array([action_true_rewards[(a, r)] for a in action_ids]).reshape(n_samples * n_batches) for r in range(n_rewards)
]
groups = np.random.randint(n_groups, size=n_samples * n_batches)
action_costs = {action: np.random.rand() for action in unique_actions}
costs = np.array([action_costs[a] for a in action_ids])
context = np.random.rand(n_samples * n_batches, n_features)
action_propensity_score = {action: np.random.rand() for action in unique_actions}
propensity_score = np.array([action_propensity_score[a] for a in action_ids])
df = pd.DataFrame(
    {
        "batch": batches,
        "action_id": action_ids,
        "cost": costs,
        "group": groups,
        **{f"reward_{r}": rewards[r] for r in range(n_rewards)},
        **{f"true_reward_{r}": true_rewards[r] for r in range(n_rewards)},
        **{f"context_{i}": context[:, i] for i in range(n_features)},
        "propensity_score": propensity_score,
    }
)
contextual_features = [col for col in df.columns if col.startswith("context")]

## Generate Model

Using the cold_start method of CmabBernoulliCC, we can create a model to be used for offline policy evaluation.

In [4]:
action_ids_cost = {action_id: df["cost"][df["action_id"] == action_id].iloc[0] for action_id in unique_actions}

mab = CmabBernoulliCC.cold_start(action_ids_cost=action_ids_cost, n_features=len(contextual_features))

## OPE

Given the model and the OPE data from the logging policy, we can either evaluate the model using the logging policy, or update it with the logging policy data prior to the evaluation.

In [5]:
evaluator = OfflinePolicyEvaluator(
    split_prop=0.5,
    n_trials=10,
    fast_fit=True,
    scaler=MinMaxScaler(),
    ope_estimators=None,
    verbose=True,
    propensity_score_model_type="batch_empirical",
    expected_reward_model_type="gbm",
    importance_weights_model_type="logreg",
    batch_feature="batch",
    action_feature="action_id",
    reward_feature="reward_0",
    true_reward_feature="true_reward_0",
    contextual_features=contextual_features,
    group_feature="group",
    cost_feature="cost",
    propensity_score_feature="propensity_score",
)

In [6]:
evaluator.evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 284.95it/s]


2026-06-29 09:48:15.108 | INFO     | pybandits.offline_policy_evaluator:_estimate_propensity_score:903 - Data batch-empirical estimation of propensity score.


2026-06-29 09:48:15.116 | INFO     | pybandits.offline_policy_evaluator:_estimate_expected_reward:952 - Data prediction of expected reward based on gbm model.


2026-06-29 09:48:16.583 | INFO     | pybandits.offline_policy_evaluator:estimate_policy:1069 - Data prediction of expected policy based on Monte Carlo experiments using 4 cores.


...

2026-06-29 09:48:46.515 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'sg-dr' for reward 'reward_0'.


2026-06-29 09:48:46.817 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-06-29 09:48:47.119 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.493836,0.461211,0.527580,0.017026,b-ipw,reward_0
1,0.488280,0.487925,0.488635,0.000180,dm,reward_0
2,0.492344,0.460304,0.524988,0.016570,dr,reward_0
3,0.488280,0.487938,0.488636,0.000178,dros-opt,reward_0
4,0.492344,0.459623,0.524687,0.016491,dros-pess,reward_0
5,0.492015,0.458349,0.526035,0.017333,ipw,reward_0
6,0.492315,0.460021,0.526674,0.017150,rep,reward_0
7,0.492346,0.459949,0.524824,0.016459,sndr,reward_0
8,0.492364,0.459837,0.526197,0.017154,snips,reward_0
9,0.492344,0.460056,0.524470,0.016395,sg-dr,reward_0


In [7]:
evaluator.update_and_evaluate(mab=mab, logged_data=df, visualize=True, n_mc_experiments=1000)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 293.91it/s]


2026-06-29 09:48:47.571 | INFO     | pybandits.offline_policy_evaluator:_update_mab:1301 - Offline policy update for <class 'pybandits.cmab.CmabBernoulliCC'>.


SVI:   0%|          | 0/1000 [00:00<?, ?it/s]

SVI:   0%|          | 1/1000 [00:00<07:56,  2.10it/s]

...

2026-06-29 09:49:39.609 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'sg-dr' for reward 'reward_0'.


2026-06-29 09:49:39.917 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'sg-ipw' for reward 'reward_0'.


2026-06-29 09:49:40.221 | INFO     | pybandits.offline_policy_evaluator:_evaluate:1187 - Running OPE estimator 'switch-dr' for reward 'reward_0'.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.505066,0.458545,0.555044,0.024387,b-ipw,reward_0
1,0.488272,0.487902,0.488631,0.000185,dm,reward_0
2,0.497641,0.458392,0.538296,0.020258,dr,reward_0
3,0.488272,0.487908,0.488639,0.000185,dros-opt,reward_0
4,0.497641,0.457332,0.536729,0.020300,dros-pess,reward_0
5,0.497086,0.450323,0.546149,0.024047,ipw,reward_0
6,0.497668,0.452101,0.545307,0.023801,rep,reward_0
7,0.497653,0.456409,0.536608,0.020449,sndr,reward_0
8,0.497698,0.452408,0.544565,0.023646,snips,reward_0
9,0.497641,0.457737,0.537664,0.020271,sg-dr,reward_0
